# Chip Orthorectification — Ground-Extent ROI Extraction and Projection

**Objective**: Extract a chip of specified ground extent (in meters) centered on a lat/lon point, then orthorectify to a local ENU meter grid.

## Modules Used
- `grdl.IO` — SICD/SIDD reader
- `grdl.geolocation` — Pixel ↔ ground transforms
- `grdl.geolocation.coordinates` — `enu_to_geodetic()`
- `grdl.image_processing.ortho` — `ENUGrid`, `orthorectify()`

## Preamble

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(f"GRDL {required}+ required. Found {current}.")

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.coordinates import enu_to_geodetic
from grdl.image_processing.ortho import orthorectify, ENUGrid

## Configuration

In [ ]:
SICD_PATH = Path("/path/to/your/sicd_file.nitf")
CENTER_LAT = 34.05  # or None for center pixel
CENTER_LON = -118.25
EXTENT_M = 500.0  # Ground extent in meters
PIXEL_SIZE_M = 1.0  # Output pixel size in meters

assert SICD_PATH.exists(), f"File not found: {SICD_PATH}"
print(f"✓ SICD: {SICD_PATH.name}")

## Step 1: Compute Pixel Extent from Ground Extent

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    geo = create_geolocation(reader)
    
    if CENTER_LAT is not None:
        center_row, center_col = geo.latlon_to_image(CENTER_LAT, CENTER_LON)[0:2]
    else:
        center_row = meta.image_data.num_rows // 2
        center_col = meta.image_data.num_cols // 2
    
    # Approximate pixel extent (simple approach)
    half_extent = EXTENT_M / 2.0
    pixel_spacing_m = meta.grid.row.ss  # row sample spacing
    chip_size = int(EXTENT_M / pixel_spacing_m)
    
    r_start = max(0, int(center_row - chip_size//2))
    c_start = max(0, int(center_col - chip_size//2))
    r_end = min(meta.image_data.num_rows, r_start + chip_size)
    c_end = min(meta.image_data.num_cols, c_start + chip_size)
    
    chip = reader.read(row_start=r_start, row_end=r_end, col_start=c_start, col_end=c_end)

print(f"Extracted chip: {chip.shape}")

## Step 2: Define ENU Grid

In [ ]:
from grdl.geolocation.chip import ChipGeolocation

with get_reader('sicd', SICD_PATH) as reader:
    geo_full = create_geolocation(reader)
    geo = ChipGeolocation(geo_full, row_offset=r_start, col_offset=c_start)
    
    center_lat, center_lon, _ = geo.image_to_latlon(chip.shape[0]//2, chip.shape[1]//2)

enu_grid = ENUGrid.from_center(
    center_lat=center_lat,
    center_lon=center_lon,
    width_m=EXTENT_M,
    height_m=EXTENT_M,
    pixel_size_m=PIXEL_SIZE_M,
)

print(f"ENU Grid: {enu_grid.shape}, pixel size: {enu_grid.pixel_size_m} m")

## Step 3: Orthorectify to ENU Meters

In [ ]:
magnitude = np.abs(chip)
ortho_result = orthorectify(magnitude, geo, enu_grid, interpolation='bilinear')
ortho_magnitude = ortho_result.data

print(f"✓ Orthorectified: {ortho_magnitude.shape}")

## Step 4: Visualization

In [ ]:
viewer = napari.Viewer(title="Chip Ortho to ENU")

mag_db = 20*np.log10(magnitude + 1e-12)
ortho_db = 20*np.log10(ortho_magnitude + 1e-12)

viewer.add_image(mag_db, name="Slant Range Chip", colormap="gray")
viewer.add_image(ortho_db, name="ENU Ortho (meters)", colormap="gray", visible=True)

print("✓ Napari viewer launched")
print(f"  ENU grid: East-North-Up, pixel size = {PIXEL_SIZE_M} m")

## Summary

✅ Ground-extent chip extraction
✅ ENU meter grid orthorectification
✅ Side-by-side slant vs ground-meter visualization